In [1]:
# imports
import pandas as pd
import matplotlib.pyplot as plt

from finvizfinance.quote import finvizfinance

import os

In [2]:
# directory path
dir_path = r"..\14_stocks_analysis\00_data"
if not os.path.isdir(dir_path):
    raise FileNotFoundError(f'Directory does not exist: {dir_path}')
print(f'Using data directory: {dir_path}')

Using data directory: ..\14_stocks_analysis\00_data


In [3]:
# concatenate all tabular files into a single dataframe
files = [
    file for file in os.listdir(dir_path)
    if file.lower().endswith(('.csv', '.xlsx', '.xls'))
]
if not files:
    raise ValueError(f'No CSV/XLSX/XLS files found in: {dir_path}')

def load_table(file_name):
    file_path = os.path.join(dir_path, file_name)
    if file_name.lower().endswith('.csv'):
        return pd.read_csv(file_path)
    return pd.read_excel(file_path)

# file name have format like 2603, 2602..first two digits are year and last two are month
# when concatenating, we need to add a column with the date extracted from the file name
def extract_date(file_name):
    year = int(file_name[:2]) + 2000
    month = int(file_name[2:4])
    return pd.Timestamp(year=year, month=month, day=1)

df = pd.concat([load_table(file).assign(date=extract_date(file)) for file in sorted(files)], ignore_index=True)

# lowercase column names
df.columns = df.columns.str.lower()

# replace " " with "_" in column names
df.columns = df.columns.str.replace(" ", "_")

df.head()

,unnamed:_0,symbol,stock,market_cap,price,fair_value_(%),z-score,f-score,m-score,value_generation,date
0,1.0,Lululemon Athletica Inc.,Lululemon Athletica Inc.,20.11B,168.18,116.31,7.43,8.0,-2.78,Robust,2025-11-01
1,NaN,LULU,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-11-01
2,2.0,"PayPal Holdings, Inc.","PayPal Holdings, Inc.",58.63B,60.57,100.92,1.96,7.0,-2.99,Robust,2025-11-01
3,NaN,PYPL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-11-01
4,3.0,Adobe Inc.,Adobe Inc.,139.08B,324.19,82.13,8.92,7.0,-2.49,Robust,2025-11-01


In [4]:
# create a copy of symbol column named 'symbol_copy'
df['symbol_copy'] = df['symbol']

In [5]:
# the series in df.symbol_copy, should be shifted backwards by 1 row, to align with the stock name in the same row
df['symbol_copy'] = df['symbol_copy'].shift(-1)

In [6]:
# drop nan in df.unnamed:_0
df = df.dropna(subset=['unnamed:_0'])

In [7]:
# drop columns 'unnamed: 0', 'symbol'
df = df.drop(columns=['unnamed:_0', 'symbol'])

# rename column 'symbol_copy' to 'symbol'
df = df.rename(columns={'symbol_copy': 'symbol'})

# columns order 'date', 'symbol', 'stock', 'market cap', 'price', 'fair value (%)', 'z-score', 'f-score', 'm-score', 'value generation'
df = df[['date', 'symbol', 'stock', 'market_cap', 'price', 'fair_value_(%)', 'z-score', 'f-score', 'm-score', 'value_generation']]

In [8]:
df.columns

Index(['date', 'symbol', 'stock', 'market_cap', 'price', 'fair_value_(%)',
       'z-score', 'f-score', 'm-score', 'value_generation'],
      dtype='object')

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 250 entries, 0 to 498
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   date              250 non-null    datetime64[us]
 1   symbol            250 non-null    object        
 2   stock             250 non-null    object        
 3   market_cap        250 non-null    object        
 4   price             250 non-null    float64       
 5   fair_value_(%)    250 non-null    float64       
 6   z-score           250 non-null    float64       
 7   f-score           250 non-null    object        
 8   m-score           250 non-null    float64       
 9   value_generation  250 non-null    object        
dtypes: datetime64[us](1), float64(4), object(5)
memory usage: 21.5+ KB


In [10]:
# create a copy of the dataframe
df_copy = df.copy()

# keep only date, symbol columns
df_copy = df_copy[['date', 'symbol']]

# group by symbol and count the number of occurrences of each symbol
symbol_counts = df_copy.groupby('symbol').size().reset_index(name='count')

# sort the dataframe by count in descending order
symbol_counts = symbol_counts.sort_values(by='count', ascending=False)

# display the top 10 symbols with the most occurrences
print(symbol_counts.head(10))

   symbol  count
2    ADBE      5
18    CLX      5
41   FTNT      5
93   PYPL      5
20    CMG      4
7     AVY      4
17     CL      4
35     EW      4
50    HUM      4
25   DECK      4


In [11]:
# drop duplicates in df.symbol, keep last occurrence
df = df.drop_duplicates(subset='symbol', keep='last')

# merge df with symbol_counts on symbol, keeping only rows that are in df
df = df.merge(symbol_counts, on='symbol', how='inner')

# sort df by count in descending order
df = df.sort_values(by='count', ascending=False)

df.head(10)

,date,symbol,stock,market_cap,price,fair_value_(%),z-score,f-score,m-score,value_generation,count
83,2026-03-01,PYPL,"PayPal Holdings, Inc.",42.66B,444859.00,87.65,1.99,8.0,-2.54,Robust,5
72,2026-03-01,CLX,The Clorox Company,12.44B,102.28,87.39,2.44,6.0,-3.16,Robust,5
69,2026-03-01,ADBE,Adobe Inc.,98.74B,2408252.00,101.55,7.53,7.0,-3.41,Robust,5
117,2026-03-01,FTNT,"Fortinet, Inc.",59.24B,79725.00,17.65,5.22,7.0,-2.23,Robust,5
39,2026-02-01,CMG,"Chipotle Mexican Grill, Inc.",48.27B,36.71,50.78,7.74,8.0,-1.70,Robust,4
42,2026-02-01,AVY,Avery Dennison Corporation,14.49B,187.20,29.80,3.93,9.0,-2.89,Robust,4
41,2026-02-01,DECK,Deckers Outdoor Corporation,15.98B,108.74,33.70,14.21,9.0,-2.89,Robust,4
55,2026-02-01,TMO,Thermo Fisher Scientific Inc.,192.77B,512.69,12.11,4.07,8.0,-2.65,Robust,4
66,2026-02-01,JKHY,"Jack Henry & Associates, Inc.",12.20B,168.43,23.66,10.78,8.0,-2.31,Resilient,4
74,2026-03-01,LDOS,"Leidos Holdings, Inc.",20.35B,153.66,78.00,4.15,8.0,-2.51,Robust,4


In [12]:
# market cap column is object. We need to convert it to numeric, removing any non-numeric characters ("T" indicating trillions, 
# "B", indicating billions, "M" indicating millions)
# These characters are at the end of the string, so we can use str.replace to remove them and then convert to numeric
# They will be added in a new column named 'market cap um'

# take last character of market cap column and add a new column 'market_cap_um' with the value of the last character
df['market_cap_um'] = df['market_cap'].str[-1]

# replace 'T', 'B', 'M' with "" in market cap column and convert to numeric
df['market_cap'] = df['market_cap'].str.replace(r'[TBM]', '', regex=True).astype(float)


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 119 entries, 83 to 118
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   date              119 non-null    datetime64[us]
 1   symbol            119 non-null    object        
 2   stock             119 non-null    object        
 3   market_cap        119 non-null    float64       
 4   price             119 non-null    float64       
 5   fair_value_(%)    119 non-null    float64       
 6   z-score           119 non-null    float64       
 7   f-score           119 non-null    object        
 8   m-score           119 non-null    float64       
 9   value_generation  119 non-null    object        
 10  count             119 non-null    int64         
 11  market_cap_um     119 non-null    object        
dtypes: datetime64[us](1), float64(5), int64(1), object(5)
memory usage: 12.1+ KB


In [14]:
# if market cap um is M, market cap / 1000, if market cap um is T, market cap * 1000, if market cap um is B, market cap / 1
def convert_market_cap(row):
    if row['market_cap_um'] == 'M':
        return row['market_cap'] / 1000
    elif row['market_cap_um'] == 'T':
        return row['market_cap'] * 1000
    elif row['market_cap_um'] == 'B':
        return row['market_cap']
    else:
        return row['market_cap']

# apply the function to the dataframe and create a new column 'market_cap_converted'
df['market_cap_converted'] = df.apply(convert_market_cap, axis=1)

In [15]:
# drop market cap	
df = df.drop(columns=['market_cap'])

# rename market cap converted to market cap
df = df.rename(columns={'market_cap_converted': 'market_cap'})

# columns order 'date', 'symbol', 'stock', 'price', 'fair value (%)', 'z-score', 'f-score', 'm-score', 'value generation', 'market cap um', 'market cap'
df = df[['date', 'symbol', 'stock', 'price', 'fair_value_(%)', 'z-score', 
'f-score', 'm-score', 'value_generation', 'market_cap_um', 'market_cap', 'count']]


In [16]:
# --- market cap filter ---
# drop all rows where market cap um is M
df = df[df['market_cap_um'] != 'M']

# drop all rows where market cap um is B and market cap is less than 10
df = df[~((df['market_cap_um'] == 'B') & (df['market_cap'] < 10))]

# rename df.market_cap to market_cap_(B)
df = df.rename(columns={'market_cap': 'market_cap_(B)'})

# drop market_cap_um
df = df.drop(columns=['market_cap_um'])

In [17]:
# add currency column according to the stock exchange where the stock is listed. Need to identify currency based on df.symbol 
# ends with:
# .MI -> EUR
# .DE -> EUR
# .PA -> EUR
# .L -> GBP
# .T -> JPY
# .AX -> AUD
# .SR -> SAR
# .HK -> HKD
# .TO -> CAD
# .NZ -> NZD
# .KL -> MYR
# .MX -> MXN
# if df.symbol does not contain ".", add USD as currency
# if none of the above conditions are met, currency 
def identify_currency(symbol):
    if symbol.endswith('.MI') or symbol.endswith('.DE') or symbol.endswith('.PA'):
        return 'EUR'
    elif symbol.endswith('.L'):
        return 'GBP'
    elif symbol.endswith('.T'):
        return 'JPY'
    elif symbol.endswith('.AX'):
        return 'AUD'
    elif symbol.endswith('.SR'):
        return 'SAR'
    elif symbol.endswith('.HK'):
        return 'HKD'
    elif symbol.endswith('.TO'):
        return 'CAD'
    elif symbol.endswith('.NZ'):
        return 'NZD'
    elif symbol.endswith('.KL'):
        return 'MYR'
    elif symbol.endswith('.MX'):
        return 'MXN'
    elif '.' not in symbol:
        return 'USD'
    else:
        return 'Unknown'
    
# apply the function to the dataframe and create a new column 'currency'
df['currency'] = df['symbol'].apply(identify_currency)

In [18]:
# print Unknown symbols in currency column
print(df[df['currency'] == 'Unknown']['symbol'])

Series([], Name: symbol, dtype: object)


In [19]:
# --- currency filter ---
# keep only EUR and USD stocks
df = df[df['currency'].isin(['EUR', 'USD'])]

In [20]:
# --- solidity filter ---
# convert z-score, f-score, m-score to numeric
df['z-score'] = pd.to_numeric(df['z-score'], errors='coerce')
df['f-score'] = pd.to_numeric(df['f-score'], errors='coerce')
df['m-score'] = pd.to_numeric(df['m-score'], errors='coerce')

# drop all rows where z-score is less than 1.81
# drop all rows where f-score is less than 3
# drop all rows where m-score is higher than -1.78
df = df[~((df['z-score'] < 1.81) | (df['f-score'] < 3) | (df['m-score'] > -1.78))]

In [21]:
# --- fair_value_(%) filter ---
# convert fair_value_(%) to numeric
df['fair_value_(%)'] = pd.to_numeric(df['fair_value_(%)'], errors='coerce')

# keep only rows where fair_value_(%) is higher than 0
df = df[df['fair_value_(%)'] > 0]

In [22]:
# list of tickers in the dataframe
tickers = df['symbol'].tolist()

In [23]:
# list of tickers in the dataframe
tickers = df['symbol'].tolist()

# TODO: change it later, we will keep first 3 tickers for testing
# tickers = tickers[:3]

tickers

['PYPL',
 'CLX',
 'ADBE',
 'FTNT',
 'AVY',
 'DECK',
 'TMO',
 'JKHY',
 'LDOS',
 'CL',
 'MSCI',
 'EW',
 'ZTS',
 'ALGN',
 'HUM',
 'UPS',
 'LULU',
 'J',
 'MSI',
 'IQV',
 'CI',
 'DPZ',
 'EOG',
 'HLT',
 'GNRC',
 'VRSK',
 'OTIS',
 'IDXX',
 'TSN',
 'FAST',
 'SYY',
 'INTU',
 'NWSA',
 'WRB',
 'MTD',
 'QCOM',
 'ROST',
 'ADSK',
 'PKG',
 'UNP',
 'META',
 'NFLX',
 'FFIV',
 'KMB',
 'NWS',
 'ROK',
 'MA',
 'UNH',
 'LEN',
 'IT',
 'TT',
 'BF-B',
 'CHD',
 'PHM',
 'CRM',
 'GD',
 'TSCO',
 'WAB',
 'ACN',
 'HES',
 'BBY',
 'FCX',
 'FOXA',
 'MDT',
 'HSY',
 'NOC',
 'MPC',
 'MAR',
 'MO',
 'MDLZ',
 'LVS',
 'LH',
 'STE',
 'WST',
 'BR',
 'DXCM',
 'EFX',
 'CTSH',
 'PPG',
 'RMD',
 'ROL',
 'URI',
 'DHI',
 'MRSH',
 'CARR',
 'EQT',
 'NKE',
 'ANET']

In [24]:
stock = finvizfinance(tickers[0])

In [25]:
# #  --- chart printing ---
# # keep only for last selected tickers

# from pathlib import Path
# from IPython.display import Image, display

# ticker = tickers[0]
# output_dir = Path('asset')
# output_dir.mkdir(parents=True, exist_ok=True)

# # Generate the chart image from Finviz
# stock.ticker_charts(out_dir=str(output_dir))

# candidate_paths = [
#     output_dir / f'{ticker}.jpg',
#     output_dir / f'{ticker}.jpeg',
#     Path(f'{ticker}.jpg'),
#     Path(f'{ticker}.jpeg'),
# ]
# image_path = next((p for p in candidate_paths if p.exists()), None)

# if image_path is not None:
#     display(Image(filename=str(image_path)))
# else:
#     print(f'Chart image not found for {ticker}. Checked: {candidate_paths}')

In [26]:
stock_fundament = stock.ticker_fundament()

In [27]:
print(stock_fundament)

{'Company': 'PayPal Holdings Inc', 'Sector': 'Financial', 'Industry': 'Credit Services', 'Country': 'USA', 'Exchange': 'NASD', 'Index': 'NDX, S&P 500', 'P/E': '8.38', 'EPS (ttm)': '5.41', 'Insider Own': '-', 'Shs Outstand': '920.00M', 'Perf Week': '0.31%', 'Market Cap': '41.74B', 'Forward P/E': '7.86', 'EPS next Y': '5.77', 'Insider Trans': '-', 'Shs Float': '919.33M', 'Perf Month': '-3.02%', 'Enterprise Value': '43.42B', 'PEG': '1.24', 'EPS next Q': '1.27', 'Inst Own': '79.25%', 'Short Float': '4.70%', 'Perf Quarter': '-22.34%', 'Income': '5.23B', 'P/S': '1.25', 'EPS this Y': '-0.13%', 'Inst Trans': '-0.25%', 'Short Ratio': '1.80', 'Perf Half Y': '-31.98%', 'Sales': '33.34B', 'P/B': '2.06', 'EPS next Y Percentage': '8.83%', 'ROA': '6.47%', 'Short Interest': '43.19M', 'Perf YTD': '-22.34%', 'Book/sh': '22.02', 'P/C': '4.01', 'EPS next 5Y': '6.33%', 'ROE': '25.73%', '52W High': '79.50 -42.97%', 'Perf Year': '-32.48%', 'Cash/sh': '11.32', 'P/FCF': '7.50', 'EPS past 3/5Y': '37.29% 8.84%',

In [28]:
stock_fundament

{'Company': 'PayPal Holdings Inc',
 'Sector': 'Financial',
 'Industry': 'Credit Services',
 'Country': 'USA',
 'Exchange': 'NASD',
 'Index': 'NDX, S&P 500',
 'P/E': '8.38',
 'EPS (ttm)': '5.41',
 'Insider Own': '-',
 'Shs Outstand': '920.00M',
 'Perf Week': '0.31%',
 'Market Cap': '41.74B',
 'Forward P/E': '7.86',
 'EPS next Y': '5.77',
 'Insider Trans': '-',
 'Shs Float': '919.33M',
 'Perf Month': '-3.02%',
 'Enterprise Value': '43.42B',
 'PEG': '1.24',
 'EPS next Q': '1.27',
 'Inst Own': '79.25%',
 'Short Float': '4.70%',
 'Perf Quarter': '-22.34%',
 'Income': '5.23B',
 'P/S': '1.25',
 'EPS this Y': '-0.13%',
 'Inst Trans': '-0.25%',
 'Short Ratio': '1.80',
 'Perf Half Y': '-31.98%',
 'Sales': '33.34B',
 'P/B': '2.06',
 'EPS next Y Percentage': '8.83%',
 'ROA': '6.47%',
 'Short Interest': '43.19M',
 'Perf YTD': '-22.34%',
 'Book/sh': '22.02',
 'P/C': '4.01',
 'EPS next 5Y': '6.33%',
 'ROE': '25.73%',
 '52W High': '79.50 -42.97%',
 'Perf Year': '-32.48%',
 'Cash/sh': '11.32',
 'P/FCF'

In [29]:
stock_fundament.keys()

dict_keys(['Company', 'Sector', 'Industry', 'Country', 'Exchange', 'Index', 'P/E', 'EPS (ttm)', 'Insider Own', 'Shs Outstand', 'Perf Week', 'Market Cap', 'Forward P/E', 'EPS next Y', 'Insider Trans', 'Shs Float', 'Perf Month', 'Enterprise Value', 'PEG', 'EPS next Q', 'Inst Own', 'Short Float', 'Perf Quarter', 'Income', 'P/S', 'EPS this Y', 'Inst Trans', 'Short Ratio', 'Perf Half Y', 'Sales', 'P/B', 'EPS next Y Percentage', 'ROA', 'Short Interest', 'Perf YTD', 'Book/sh', 'P/C', 'EPS next 5Y', 'ROE', '52W High', 'Perf Year', 'Cash/sh', 'P/FCF', 'EPS past 3/5Y', 'ROIC', '52W Low', 'Perf 3Y', 'Dividend Est.', 'EV/EBITDA', 'Sales past 3/5Y', 'Gross Margin', 'Volatility W', 'Volatility M', 'Perf 5Y', 'Dividend TTM', 'EV/Sales', 'EPS Y/Y TTM', 'Oper. Margin', 'ATR (14)', 'Perf 10Y', 'Dividend Ex-Date', 'Quick Ratio', 'Sales Y/Y TTM', 'Profit Margin', 'RSI (14)', 'Recom', 'Dividend Gr. 3/5Y', 'Current Ratio', 'EPS Q/Q', 'SMA20', 'Beta', 'Target Price', 'Payout', 'Debt/Eq', 'Sales Q/Q', 'SMA50'

In [30]:
# all listed keys in stock_fundament
"""
dict_keys(['Company', 'Sector', 'Industry', 'Country', 'Exchange', 'Index', 'P/E', 'EPS (ttm)', 
'Insider Own', 'Shs Outstand', 'Perf Week', 'Market Cap', 'Forward P/E', 'EPS next Y', 'Insider Trans', 'Shs Float', 'Perf Month', 
'Enterprise Value', 'PEG', 'EPS next Q', 'Inst Own', 'Short Float', 'Perf Quarter', 'Income', 'P/S', 'EPS this Y', 'Inst Trans', 
'Short Ratio', 'Perf Half Y', 'Sales', 'P/B', 'EPS next Y Percentage', 'ROA', 'Short Interest', 'Perf YTD', 'Book/sh', 'P/C', 
'EPS next 5Y', 'ROE', '52W High', 'Perf Year', 'Cash/sh', 'P/FCF', 'EPS past 3/5Y', 'ROIC', '52W Low', 'Perf 3Y', 'Dividend Est.', 
'EV/EBITDA', 'Sales past 3/5Y', 'Gross Margin', 'Volatility W', 'Volatility M', 'Perf 5Y', 'Dividend TTM', 'EV/Sales', 'EPS Y/Y TTM', 
'Oper. Margin', 'ATR (14)', 'Perf 10Y', 'Dividend Ex-Date', 'Quick Ratio', 'Sales Y/Y TTM', 'Profit Margin', 'RSI (14)', 'Recom', 
'Dividend Gr. 3/5Y', 'Current Ratio', 'EPS Q/Q', 'SMA20', 'Beta', 'Target Price', 'Payout', 'Debt/Eq', 'Sales Q/Q', 'SMA50', 'Rel Volume', 
'Prev Close', 'Employees', 'LT Debt/Eq', 'Earnings', 'SMA200', 'Avg Volume', 'Price', 'IPO', 'Option/Short', 'EPS/Sales Surpr.', 'Trades', 'Volume', 'Change'])
""";

# keys to keep in stock_fundament
keys_to_keep = ['Company', 'Sector', 'Industry', 'P/E', 'EPS (ttm)', 
'Perf Week', 'Forward P/E', 'EPS next Y', 'Perf Month', 
'Short Float', 'Perf Quarter', 'P/S', 'EPS this Y',  
'Short Ratio', 'Perf Half Y', 'P/B', 'ROA', 'Perf YTD', 'P/C', 
'ROE', '52W High', 'Perf Year', 'P/FCF', 'EPS past 3/5Y', 'ROIC', '52W Low', 'Perf 3Y', 'Dividend Est.', 
'EV/EBITDA', 'Sales past 3/5Y', 'Volatility W', 'Volatility M', 'Perf 5Y', 'Dividend TTM', 'EV/Sales', 'EPS Y/Y TTM', 
'Oper. Margin', 'ATR (14)', 'Perf 10Y', 'Quick Ratio', 'Sales Y/Y TTM', 'Profit Margin', 'RSI (14)', 'Recom', 
'Dividend Gr. 3/5Y', 'Current Ratio', 'EPS Q/Q', 'SMA20', 'Beta', 'Payout', 'Debt/Eq', 'Sales Q/Q', 'SMA50', 'Rel Volume', 
'Prev Close', 'LT Debt/Eq', 'SMA200', 'Avg Volume', 'Price', 'IPO', 'EPS/Sales Surpr.', 'Trades', 'Volume', 'Change']

# TODO: 'Perf Month', 'Perf Quarter', 'Perf YTD', 'Perf Half Y', 'Perf Year', 'Perf 3Y', 'Perf 5Y', 'Perf 10Y' should be converted to numeric, removing the "%" character, 
# then used as one of the first filters to keep only stocks with negative performance

# TODO: 'EPS past 3/5Y': '37.29% 8.84%' should be split into two separate columns, one for 3Y and one for 5Y, but for now we will keep it as is
# TODO: 'Sales past 3/5Y': '7.21% 9.24%' should be split into two separate columns, one for 3Y and one for 5Y, but for now we will keep it as is

stock_fundament_filtered = {key: stock_fundament[key] for key in keys_to_keep if key in stock_fundament}

print(stock_fundament_filtered)

{'Company': 'PayPal Holdings Inc', 'Sector': 'Financial', 'Industry': 'Credit Services', 'P/E': '8.38', 'EPS (ttm)': '5.41', 'Perf Week': '0.31%', 'Forward P/E': '7.86', 'EPS next Y': '5.77', 'Perf Month': '-3.02%', 'Short Float': '4.70%', 'Perf Quarter': '-22.34%', 'P/S': '1.25', 'EPS this Y': '-0.13%', 'Short Ratio': '1.80', 'Perf Half Y': '-31.98%', 'P/B': '2.06', 'ROA': '6.47%', 'Perf YTD': '-22.34%', 'P/C': '4.01', 'ROE': '25.73%', '52W High': '79.50 -42.97%', 'Perf Year': '-32.48%', 'P/FCF': '7.50', 'EPS past 3/5Y': '37.29% 8.84%', 'ROIC': '16.99%', '52W Low': '38.46 17.89%', 'Perf 3Y': '-40.29%', 'Dividend Est.': '0.31 (0.68%)', 'EV/EBITDA': '5.77', 'Sales past 3/5Y': '7.21% 9.24%', 'Volatility W': '3.09%', 'Volatility M': '3.06%', 'Perf 5Y': '-81.68%', 'Dividend TTM': '0.28 (0.62%)', 'EV/Sales': '1.30', 'EPS Y/Y TTM': '35.36%', 'Oper. Margin': '19.69%', 'ATR (14)': '1.61', 'Perf 10Y': '15.08%', 'Quick Ratio': '1.29', 'Sales Y/Y TTM': '4.88%', 'Profit Margin': '15.70%', 'RSI (14

In [31]:
# stock_fundament dict to dataframe
stock_fundament_df = pd.DataFrame.from_dict(stock_fundament, orient='index', columns=['value'])
stock_fundament_df

,value
Company,PayPal Holdings Inc
Sector,Financial
Industry,Credit Services
Country,USA
Exchange,NASD
...,...
Option/Short,Yes / Yes
EPS/Sales Surpr.,-4.43% -1.28%
Trades,\n\n
Volume,"12,723,116"


In [32]:
stock_fundament_df.T

,Company,Sector,Industry,Country,Exchange,Index,P/E,EPS (ttm),Insider Own,Shs Outstand,...,Earnings,SMA200,Avg Volume,Price,IPO,Option/Short,EPS/Sales Surpr.,Trades,Volume,Change
value,PayPal Holdings Inc,Financial,Credit Services,USA,NASD,"NDX, S&P 500",8.38,5.41,-,920.00M,...,Feb 03 BMO,-26.57%,23.99M,45.34,"Jul 06, 2015",Yes / Yes,-4.43% -1.28%,\n\n,"12,723,116",1.59%


In [33]:
stock_fundament_df.columns

Index(['value'], dtype='object')

In [34]:
# from stock_fundament deleted insiders transactions, add in later with separate function to fetch insider transactions.. inside_trader_df = stock.ticker_inside_trader()

In [35]:
# print length of dataframe
print("Number of rows in the dataframe:", len(df))

# print lenght of unique symbols in the dataframe (for comparison with the original dataframe)
print("Number of unique symbols in the dataframe:", df['symbol'].nunique())

Number of rows in the dataframe: 88
Number of unique symbols in the dataframe: 88


In [36]:
df.sample(10)

,date,symbol,stock,price,fair_value_(%),z-score,f-score,m-score,value_generation,market_cap_(B),count,currency
69,2026-03-01,ADBE,Adobe Inc.,2408252.00,101.55,7.53,7.0,-3.41,Robust,98.74,5,USD
8,2025-11-01,HSY,The Hershey Company,186.00,5.43,4.71,7.0,-2.32,Robust,0.00,1,USD
25,2026-01-01,VRSK,"Verisk Analytics, Inc.",184.68,55.18,8.36,8.0,-3.25,Robust,25.76,3,USD
37,2026-01-01,CRM,"Salesforce, Inc.",189.97,63.67,3.78,8.0,-2.35,Resilient,184.12,1,USD
81,2026-03-01,BF-B,Brown-Forman Corporation,27.11,43.75,2.14,7.0,-2.30,Robust,12.82,2,USD
13,2025-12-01,KMB,Kimberly-Clark Corporation,101.00,28.51,3.95,8.0,-2.71,Robust,33.51,2,USD
46,2026-02-01,ALGN,"Align Technology, Inc.",182.16,43.57,5.23,7.0,-2.87,Robust,13.06,4,USD
11,2025-12-01,PKG,Packaging Corporation of America,208.91,3.71,4.72,8.0,-2.73,Robust,18.81,2,USD
16,2025-12-01,MDLZ,"Mondelez International, Inc.",54.64,26.92,2.59,8.0,-2.23,Robust,71.22,1,USD
83,2026-03-01,PYPL,"PayPal Holdings, Inc.",444859.00,87.65,1.99,8.0,-2.54,Robust,42.66,5,USD


In [37]:
# # Examples from finvizfinance documentation
# stock = finvizfinance('tsla')

# stock.TickerCharts(out_dir='asset')

# stock_fundament = stock.ticker_fundament()
# stock_description = stock.ticker_description()
# outer_ratings_df = stock.ticker_outer_ratings()
# news_df = stock.ticker_news()
# inside_trader_df = stock.ticker_inside_trader()